# Visual-awareness probes

Compare seven ways of restricting the visual input of Qwen3-VL-8B-Instruct. Every condition uses the same image, crop window, and localization prompt. The full-image condition targets the whole canvas; the other six target the selected window.

Install the [repository environment](../../README.md#installation), then select its Python kernel. Inference needs a CUDA GPU and downloads the model on first use. Set `LOOK_NOTEBOOK_PREVIEW=1` before starting Jupyter to inspect the samples and configuration without loading a model.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code/vendor/ms-swift").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "code"))
WORK = Path(os.environ.get("WORK", ROOT / "work")).expanduser().resolve()
os.environ["WORK"] = str(WORK)
os.environ.setdefault("HF_HOME", str(WORK / "cache/huggingface"))
os.environ.setdefault("MODELSCOPE_CACHE", str(WORK / "cache/modelscope"))
os.environ.setdefault("MAX_PIXELS", "1003520")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
RUN_INFERENCE = os.environ.get("LOOK_NOTEBOOK_PREVIEW", "0") != "1"
from notebook_utils import load_samples, display_samples, display_table
SAMPLES = load_samples(ROOT)
print(f"{len(SAMPLES)} bundled RefCOCO test samples; inference={RUN_INFERENCE}")

## 1. Images and configuration

The small image set is bundled with the repository. Increase `N_IMAGES` or pass your own image records to evaluate more examples. `ADAPTER=None` tests the pretrained model; set it to a local SFT or LOOK adapter directory to compare a fine-tuned model.

In [ ]:
MODEL = "Qwen/Qwen3-VL-8B-Instruct"
ADAPTER = None
N_IMAGES = 3
SEED = 0
OUTPUT = WORK / "notebooks/visual_awareness_probes"
SELECTED = SAMPLES[:N_IMAGES]
display_samples(SELECTED)

## 2. Probe conditions

Black/grey masks keep the full image grid. Zero/mean masks act after patch embedding and retain positional information for the excluded patches. Crop-reset and crop-restored remove excluded patches entirely. The restored condition applies the same full-image vision and LLM positional overrides used by LOOK.

In [ ]:
from notebook_utils import PROBE_CONDITIONS
CONDITIONS = list(PROBE_CONDITIONS)
display_table([{"condition": name, "input": PROBE_CONDITIONS[name]} for name in CONDITIONS])

## 3. Run the model

This calls the same crop sampler, embedding-mask hook, position overrides, box parser, and IoU scorer as the evaluation scripts. It saves each raw answer, target, predicted box, and score under `OUTPUT`.

In [ ]:
from notebook_utils import run_visual_probes, summarize_probes
probe_rows = []
if RUN_INFERENCE:
    probe_rows = run_visual_probes(SELECTED, OUTPUT, model_id=MODEL, adapter=ADAPTER,
                                  seed=SEED, conditions=CONDITIONS)
else:
    print("Preview mode: model inference is disabled. Unset LOOK_NOTEBOOK_PREVIEW to run the probes.")

## 4. Compare localization

Report parse rate and IoU separately. `iou_all` counts unparseable answers as zero; `iou_parsed` considers parsed answers only. `whole_canvas_iou` is the constant full-canvas prediction on the same targets.

In [ ]:
summary = summarize_probes(probe_rows)
display_table(summary)
if summary:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.bar([r["condition"] for r in summary], [r["iou_all"] for r in summary], color="#147d83")
    ax.set_ylabel("Mean IoU (parse failures = 0)")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()

In [ ]:
from notebook_utils import display_probe_boxes
display_probe_boxes(probe_rows)


## 5. Inspect individual answers

The `full` condition should return `[0, 0, 1000, 1000]`. Other boxes are measured against the original canvas, including when the model receives only the crop.

In [ ]:
display_table(probe_rows, ["id", "condition", "gt", "pred", "iou", "answer"])
print("Saved outputs:", OUTPUT)